<div style="border:solid green 2px; padding: 20px"> <h1 style="color:green; margin-bottom:20px">Reviewers comment v1</h1>

Hello George!

I'm happy to review your project today 🙌

My name is **Justino Imbert** ([this](https://hub.tripleten.com/u/125e88ae) is my Hub profile) and today I'll be reviewing your project!


You can find my comments under the heading **«Review»**. I will categorize my comments in green, blue or red boxes like this:

<div class="alert alert-success">
    <b>Success:</b> if everything is done successfully
</div>
<div class="alert alert-warning">
    <b>Remarks:</b> if I can give some recommendations or ways to improve the project
</div>
<div class="alert alert-danger">
    <b>Needs fixing:</b> if the block requires some corrections. Work cant be accepted with the red comments
</div>

Please dont remove my comments :) If you have any questions dont hesitate to respond to my comments in a different section. 
<div class="alert alert-info"> <b>Student comments:</b> For example like this</div>   


<div class="alert alert-block alert-info">
<b>Reviewer's comment v1:</b> </a>

Amazing job with this submission! I'm approving this project!

Congrats and I wish you the best of luck in the following sprints!

Looking forward to reviewing your future work!
    
</div>


# 0. Imports & Globals

In [1]:
import pandas as pd 
import numpy as np 

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
# Economic Constants
# total budget for 200 wells (USD)
budget = 100000000
# number of wells to drill  
n_drill = 200
# number of wells to explore
n_explore = 500
# price per barrel of oil (USD)
barrel_price = 4.5
# price per thousand barrels of oil (USD) because 'product' is in thousand barrels
product_price = barrel_price * 1000

# random state for reproducibility
state = np.random.RandomState(12345)
# number of bootstrap samples for profit simulation
samples = 1000


# 1. Download and Prepare the Data

In [3]:
# dataset paths
paths = {
    0: '/datasets/geo_data_0.csv',
    1: '/datasets/geo_data_1.csv',
    2: '/datasets/geo_data_2.csv'
}

# for loop to read datasets into a dictionary of dataframes
data = {}
for region, path in paths.items():
    df = pd.read_csv(path)
    data[region] = df


## Minimal EDA per Region

In [4]:
for region, df in data.items():
    print(f'\n=== Region {region} ===')
    print(df.head())
    print(df.info())
    print(df.describe())
    print('Missing values per column:')
    print(df.isna().sum())


=== Region 0 ===
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433    

### EDA Conclusions:
 There are 100,000 rows per region. With `id`, `f0`, `f1`, `f2`, and `product` columns present in each regions dataset. Furthermore there are no missing values in any columns of any of the regional datasets.

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1:</b> </a>

Awesome job with the initial exploration!
    
</div>


# 2. Train and Test the Model for Each Region

In [5]:
# function to pull features and target from each regional dataframes
def get_features_target(df):
    features = df[['f0', 'f1', 'f2']]
    target = df['product']
    return features, target

# model variables
models = {}
valid_predictions = {}
valid_targets = {}
metrics_summary = {}

In [6]:
#
for region, df in data.items():
    print(f'\n=== Region {region} ===')
    features, target = get_features_target(df)
    X_train, X_valid, y_train, y_valid = train_test_split(
        features, target, test_size=0.25, random_state=state
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)

    # save predictions and true values
    models[region] = model
    valid_predictions[region] = y_pred
    valid_targets[region] = y_valid.reset_index(drop=True)

    # Metrics
    rmse = mean_squared_error(y_valid, y_pred, squared=False)
    mae = mean_absolute_error(y_valid, y_pred)
    r2 = r2_score(y_valid, y_pred)

    metrics_summary[region] = {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mean_pred_product':np.mean(y_pred),
        'mean_true_product':np.mean(y_valid)
    }

    print(f'Average predicted product: {np.mean(y_pred):.2f}')
    print(f'Average true product: {np.mean(y_valid):.2f}')
    print(f'RMSE: {rmse:.2f}')
    print(f'MAE: {mae:.2f}')
    print(f'R²: {r2:.2f}')


=== Region 0 ===
Average predicted product: 92.59
Average true product: 92.08
RMSE: 37.58
MAE: 30.92
R²: 0.28

=== Region 1 ===
Average predicted product: 68.77
Average true product: 68.77
RMSE: 0.89
MAE: 0.72
R²: 1.00

=== Region 2 ===
Average predicted product: 95.09
Average true product: 94.75
RMSE: 39.96
MAE: 32.80
R²: 0.20


<div class="alert alert-block alert-success">
<b>Reviewer's comment v1:</b> </a>

You did a great job training and testing the model for each region!

</div>


In [7]:
# cross-validation RMSE
cv_results = {}
for region, df in data.items():
    features, target = get_features_target(df)
    model = LinearRegression()
    neg_rmse_scores = cross_val_score(
        model, features, target, 
        cv=3, 
        scoring='neg_root_mean_squared_error'
    )

    rmse_scores = -neg_rmse_scores
    cv_results[region] = rmse_scores
    print(f'\n Region {region} Cross-Validation RMSE Scores: {rmse_scores}')
    print(f'Mean CV RMSE: {rmse_scores.mean():.2f}')



 Region 0 Cross-Validation RMSE Scores: [37.93934072 37.53195287 37.60609272]
Mean CV RMSE: 37.69

 Region 1 Cross-Validation RMSE Scores: [0.8902274  0.88939711 0.89163629]
Mean CV RMSE: 0.89

 Region 2 Cross-Validation RMSE Scores: [40.17176215 40.08514246 39.91325384]
Mean CV RMSE: 40.06


### Analysis:
Region 0 has the best RMSE at 37.67 and R² score of 0.28 out of the three regions; however, Region 1 has the worst RMSE at 0.89 and R² score of 1.00. This means that while Region 0 has an average ~30-35% error rate it is 28% better at prediction than it would be to just use the mean, but Region 1 while the error rate is significatantly lower...with an R² score of 1...this strikes me as highly suspicious and not to be trusted.

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1:</b> </a>

Good analysis! Adding the cross-validation RMSE was a great touch!

</div>


# 3. Prepare for Profit Calculation

In [8]:
# Calculate the reserves volume sufficient to develop a new well without losses
min_volumen = budget / (n_drill * product_price)
print(f'\nMinimum reserves volume per well to avoid losses: {min_volumen:.2f} thousand barrels')

for region, df in data.items():
    mean_prod = df['product'].mean()
    print(f'Region {region}: mean product = {mean_prod:.2f} thousand barrels')


Minimum reserves volume per well to avoid losses: 111.11 thousand barrels
Region 0: mean product = 92.50 thousand barrels
Region 1: mean product = 68.83 thousand barrels
Region 2: mean product = 95.00 thousand barrels


### Findings:
Looking at this project and the three datasets at a 50,000 foot view, a well needs to have a reserve volume of ~111.11 thousand barrels of oil to avoid any losses in this project. When we look at the mean `product` for each region's average wells all the wells are below the 111.11 thousand barrels needed to break even. However, since we will be choosing the best 200 wells...it looks like Region 2 would be the prime region with an average well volume of 95.00 thousand barrels and Region 0 coming in slightly behind at 92.5 thousand barrels. As we "drill down" (pun intended) into the data and choose the top 200 sites for wells per region, we should be able to beat the average wells easily.

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1:</b> </a>

You did an amazing job preparing for profit calculation!

</div>


# 4. Profit Calculation Function

In [9]:
def calculate_profit(target, predictions):
    """ 
    target: pd.Series fo true product values (thousand barrels)
    predictions: np.array or pd.Series of predicted product values (thousand barrels)
    Returns: estimated profit (USD)
    """
    # Make sure indices align
    target = target.reset_index(drop=True)
    predictions = np.array(predictions)

    # Indices of top n_drill predicted
    top_indices = predictions.argsort()[-n_drill:]

    # True product for those wells
    selected_product = target.iloc[top_indices]
    total_product = selected_product.sum()  # in thousand barrels

    revenue = total_product * product_price  # in USD
    costs = budget  # in USD
    profit = revenue - costs
    
    return profit

In [10]:
region_profits = {}

for region in data.keys():
    y_valid = valid_targets[region]
    y_pred = valid_predictions[region]

    profit = calculate_profit(y_valid, y_pred)
    region_profits[region] = profit
    print(f'Region {region}: One-shot profit estimate = ${profit:,.0f}')

Region 0: One-shot profit estimate = $33,208,260
Region 1: One-shot profit estimate = $24,150,867
Region 2: One-shot profit estimate = $25,399,159


### Findings:
According to my findings it looks like Region 0 has the highest single estimate of $33,591,411 in profit. However, this doesn't consider uncertainty/sampling variability which will be looked at in the next step where I will calculate the Risks and profit for each region using Bootstrapping.

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1:</b> </a>

Nicely done! Profit estimates are looking good!

</div>


# 5. Risk and Profit Calculations for each region via Bootstrapping

In [11]:
# Bootstrap function to simulate profit distribution
def bootstrap_region_profit(target, predictions, n_samples=samples, seed=12345):
    target = target.reset_index(drop=True)
    predictions = np.array(predictions)

    rng = np.random.RandomState(seed)
    profits = []

    n_total = len(target)

    for _ in range(n_samples):
        # sample 500 indices with replacement
        sample_indices = rng.randint(0, n_total, size=n_explore)

        sample_target = target.iloc[sample_indices].reset_index(drop=True)
        sample_pred = predictions[sample_indices]

        profit = calculate_profit(sample_target, sample_pred)
        profits.append(profit)
    
    profits = np.array(profits)
    return profits

In [12]:
bootstrap_results = {}

for region in data.keys():
    profits = bootstrap_region_profit(valid_targets[region], valid_predictions[region])

    mean_profit = profits.mean()
    conf_int = np.percentile(profits, [2.5, 97.5])
    risk_of_loss = (profits < 0).mean()  # probability of negative profit

    bootstrap_results[region] = {
        'mean_profit': mean_profit,
        'conf_int': conf_int,
        'risk_of_loss': risk_of_loss
    }

    print(f'\n=== region {region} ===')
    print(f'Mean profit: ${mean_profit:,.0f}')
    print(f'95% Confidence Interval: ${conf_int[0]:,.0f} to ${conf_int[1]:,.0f}')
    print(f'Risk of Loss: {risk_of_loss * 100:.2f}%')


=== region 0 ===
Mean profit: $3,961,650
95% Confidence Interval: $-1,112,155 to $9,097,669
Risk of Loss: 6.90%

=== region 1 ===
Mean profit: $4,456,176
95% Confidence Interval: $545,859 to $8,330,682
Risk of Loss: 1.10%

=== region 2 ===
Mean profit: $3,400,269
95% Confidence Interval: $-2,080,298 to $8,688,559
Risk of Loss: 10.70%


In [13]:
valid_regions = {
    region: res for region, res in bootstrap_results.items()
    if res['risk_of_loss'] < 0.025
}

print('\nRegions with Acceptable Risk (< 2.5%):')
for region, res in valid_regions.items():
    print(f'Region {region}: Mean Profit = ${res["mean_profit"]:,.0f}, '
          f'Risk of Loss = {res["risk_of_loss"] * 100:.2f}%')

best_region = max(valid_regions, key=lambda r: valid_regions[r]['mean_profit'])
best_stats = valid_regions[best_region]

print(f'\n>>> Recommended Region: {best_region}')
print(f'Expected mean profit: ${best_stats["mean_profit"]:,.0f}')
print(f'95% Confidence Interval: ${best_stats["conf_int"][0]:,.0f} to ${best_stats["conf_int"][1]:,.0f}')
print(f'Risk of Loss: {best_stats["risk_of_loss"] * 100:.2f}%')


Regions with Acceptable Risk (< 2.5%):
Region 1: Mean Profit = $4,456,176, Risk of Loss = 1.10%

>>> Recommended Region: 1
Expected mean profit: $4,456,176
95% Confidence Interval: $545,859 to $8,330,682
Risk of Loss: 1.10%


<div class="alert alert-block alert-warning">
<b>Reviewer's comment v1:</b> </a>

It would be nice to have a visualization for the distribution of profit. It would help illustrate the profit variability more clearly!

</div>


<div class="alert alert-block alert-success">
<b>Reviewer's comment v1:</b> </a>

Other than that, you did an incredible job here! Bootstrapping is looking fine!

</div>


## Final Profit & Risk Analysis

After performing the profit simulation and risk assessment using the bootstrapping technique (1,000 resampled iterations per region), only Region 1 met the required safety threshold for investment. The project guidelines specify that acceptable regions must have a risk of loss below 2.5%, and among those, the region with the highest average profit should be selected.

**Acceptable Regions (Risk < 2.5%)**
* **Region 1**
    - Mean Profit: $4,456,176
    - Risk of Loss: 1.10%

Regions 0 and 2 exceeded the allowable risk threshold and were therefore eliminated from consideration.

**Recommended Region**

**Region 1** is the optimal choice for developing new oil wells. It combines:
* A high positive expected profit
* A low and acceptable probability of loss
* A strong upper confidence bound, indicating potential for significantly higher returns under favorable conditions

**Confidence Interval Analysis**

The **95% confidence interval** for Region 1 ranges from:
	•	Lower bound: $545,859
	•	Upper bound: $8,330,682

This interval indicates that while profit outcomes may vary due to natural uncertainty in the reserve predictions, Region 1 maintains a **positive lower limit**, demonstrating financial robustness. The wide upper range also highlights substantial upside potential without exceeding the risk tolerance specified in the project.

**Conclusion**

Based on the profitability estimation, risk analysis, and confidence interval evaluation, **Region 1 is the most financially sound and strategically appropriate location** for the development of new oil wells. Its combination of **high expected profit** and **low downside risk** makes it the preferred choice for investment and aligns with the project’s objective of maximizing returns while maintaining risk control.

<div class="alert alert-block alert-success">
<b>Reviewer's comment v1:</b> </a>

Final conclusions are clear and well structured! Nice job!

You did an excellent job all throughout this submission! You should be proud! Keep up the great work!

</div>
